# 12a Resume Emit WILDTRACK test.txt
Regenerate the MVDeTr WILDTRACK ground-plane detection output from the trained checkpoint without retraining.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

RUN_NAME = "aug_deform_trans_lr0.0007_baseR0.1_neck128_out0_alpha1.0_id0_drop0.0_dropcam0.0_worldRK4_10_imgRK12_10_2026-04-01_11-01-57"
MVDETR_COMMIT = "66cae15382066ef009780fd1891543f5d5bdc75d"
WORK = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
MVDETR = WORK / "MVDeTr"

def find_wildtrack_root():
    candidates = [p for p in INPUT_ROOT.rglob("Wildtrack") if p.is_dir()]
    for candidate in candidates:
        required = [
            candidate / "Image_subsets",
            candidate / "annotations_positions",
            candidate / "calibrations" / "intrinsic_zero",
            candidate / "calibrations" / "extrinsic",
            candidate / "rectangles.pom",
        ]
        if all(path.exists() for path in required):
            return candidate
    raise FileNotFoundError(f"No usable Wildtrack directory under {INPUT_ROOT}; candidates={candidates[:20]}")

def find_checkpoint():
    candidates = [p for p in INPUT_ROOT.rglob("MultiviewDetector.pth") if p.is_file()]
    if not candidates:
        raise FileNotFoundError(f"No MultiviewDetector.pth under {INPUT_ROOT}")
    return candidates[0]

print("input mounts", [str(p) for p in sorted(INPUT_ROOT.iterdir())])
WILDTRACK_INPUT = find_wildtrack_root()
CKPT_INPUT = find_checkpoint()
TEST_TXT = MVDETR / "logs" / "wildtrack" / RUN_NAME / "test.txt"
OUT_TXT = WORK / "test.txt"

print("checkpoint", CKPT_INPUT, "bytes", CKPT_INPUT.stat().st_size)
print("wildtrack", WILDTRACK_INPUT)
print("wildtrack sample", sorted(p.name for p in WILDTRACK_INPUT.iterdir())[:8])

In [ ]:
def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

if MVDETR.exists():
    shutil.rmtree(MVDETR)
run(["git", "clone", "https://github.com/hou-yz/MVDeTr.git", str(MVDETR)])
run(["git", "checkout", MVDETR_COMMIT], cwd=MVDETR)
run(["git", "rev-parse", "HEAD"], cwd=MVDETR)

# Kaggle P100 is sm_60; current default torch may not support it, so pin the verified CUDA wheel.
run([
    "python", "-m", "pip", "install", "-q", "--force-reinstall",
    "torch==2.4.1", "torchvision==0.19.1",
    "--index-url", "https://download.pytorch.org/whl/cu124",
])
run(["python", "-m", "pip", "install", "-q", "kornia", "opencv-python-headless", "ninja"])

# PyTorch 2.x requires scalar_type() in AT_DISPATCH; upstream MVDeTr used Tensor.type().
cu_file = MVDETR / "multiview_detector" / "models" / "ops" / "src" / "cuda" / "ms_deform_attn_cuda.cu"
text = cu_file.read_text()
text = text.replace("value.type(),", "value.scalar_type(),")
cu_file.write_text(text)

# NumPy 2.x removed legacy aliases used by the original 2021 code.
for source_file in [
    MVDETR / "multiview_detector" / "datasets" / "Wildtrack.py",
    MVDETR / "multiview_detector" / "datasets" / "MultiviewX.py",
    MVDETR / "multiview_detector" / "datasets" / "frameDataset.py",
    MVDETR / "multiview_detector" / "models" / "mvdetr.py",
]:
    text = source_file.read_text()
    text = text.replace("dtype=np.float)", "dtype=float)")
    text = text.replace("dtype=np.bool)", "dtype=np.bool_)")
    text = text.replace("np.product(", "np.prod(")
    text = text.replace("kornia.warp_perspective(", "kornia.geometry.transform.warp_perspective(")
    text = text.replace(
        "self.gt_fpath = os.path.join(self.root, 'gt.txt')",
        "self.gt_fpath = os.path.join('/kaggle/working', 'wildtrack_gt.txt')",
    )
    source_file.write_text(text)
run(["bash", "make.sh"], cwd=MVDETR / "multiview_detector" / "models" / "ops")

In [ ]:
data_root = Path.home() / "Data"
wildtrack_root = data_root / "Wildtrack"
data_root.mkdir(parents=True, exist_ok=True)
if wildtrack_root.exists() or wildtrack_root.is_symlink():
    if wildtrack_root.is_symlink() or wildtrack_root.is_file():
        wildtrack_root.unlink()
    else:
        shutil.rmtree(wildtrack_root)
os.symlink(WILDTRACK_INPUT, wildtrack_root, target_is_directory=True)

required = [
    wildtrack_root / "Image_subsets",
    wildtrack_root / "annotations_positions",
    wildtrack_root / "calibrations" / "intrinsic_zero",
    wildtrack_root / "calibrations" / "extrinsic",
    wildtrack_root / "rectangles.pom",
]
for path in required:
    assert path.exists(), path
print("linked", wildtrack_root, "->", WILDTRACK_INPUT)

In [ ]:
run_dir = MVDETR / "logs" / "wildtrack" / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(CKPT_INPUT, run_dir / "MultiviewDetector.pth")
print("resume checkpoint", run_dir / "MultiviewDetector.pth", (run_dir / "MultiviewDetector.pth").stat().st_size)

cmd = [
    "python", "main.py",
    "-d", "wildtrack",
    "--resume", RUN_NAME,
    "--num_workers", "2",
    "--batch_size", "1",
]
run(cmd, cwd=MVDETR)
assert TEST_TXT.exists() and TEST_TXT.stat().st_size > 0, TEST_TXT

In [ ]:
import hashlib

shutil.copy2(TEST_TXT, OUT_TXT)
lines = OUT_TXT.read_text().splitlines()
assert len(lines) >= 100, len(lines)
for i, line in enumerate(lines[:20], 1):
    parts = line.split()
    assert len(parts) == 3, (i, line)
    [int(part) for part in parts]
md5 = hashlib.md5(OUT_TXT.read_bytes()).hexdigest()
print("test.txt", OUT_TXT)
print("line_count", len(lines))
print("md5", md5)
print("head")
print("\n".join(lines[:10]))
print("tail")
print("\n".join(lines[-10:]))